# Module 07: Collective Wisdom, Bounded Confidence & El Farol Market
Exploration of **Scott Page's Diversity Prediction Theorem (and Ex-Ante Error Correlation), Hegselmann-Krause Bounded Confidence Opinion Dynamics vs. DeGroot Consensus, Brian Arthur's El Farol Bar Problem (1994), and Challet & Zhang's Minority Game (1997)**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# ------------------------------------------------------------------------------
# 1. DIVERSITY PREDICTION THEOREM & ERROR CORRELATION ENGINE
# ------------------------------------------------------------------------------

def simulate_dpt_ensemble(n_agents=50, indiv_sigma=10.0, error_corr=0.25, n_trials=400, seed=42):
    # Evaluates Page's ex-post algebraic identity and ex-ante correlated error expectation
    rng = np.random.default_rng(seed)
    theta_true = 100.0
    rho = max(0.0, min(0.99, error_corr))
    z_common = rng.normal(0.0, 1.0, size=(n_trials, 1))
    z_indep = rng.normal(0.0, 1.0, size=(n_trials, n_agents))
    errors = indiv_sigma * (np.sqrt(rho) * z_common + np.sqrt(1.0 - rho) * z_indep)
    preds = theta_true + errors

    mean_pred = np.mean(preds, axis=1, keepdims=True)
    e_coll = np.mean((mean_pred[:, 0] - theta_true) ** 2)
    e_indiv = np.mean(np.mean((preds - theta_true) ** 2, axis=1))
    div_pred = np.mean(np.mean((preds - mean_pred) ** 2, axis=1))

    # Single realization check for Page's exact algebraic identity
    p0 = preds[0]
    m0 = np.mean(p0)
    exact_coll = float((m0 - theta_true) ** 2)
    exact_indiv = float(np.mean((p0 - theta_true) ** 2))
    exact_div = float(np.mean((p0 - m0) ** 2))
    return e_coll, e_indiv, div_pred, (exact_coll, exact_indiv, exact_div)

# ------------------------------------------------------------------------------
# 2. HEGSELMANN-KRAUSE BOUNDED CONFIDENCE VS DEGROOT OPINION DYNAMICS
# ------------------------------------------------------------------------------

def simulate_hk_opinions(n_agents=80, confidence_eps=0.18, steps=30, seed=42):
    # Evolves continuous opinions x_i in [0, 1] under Hegselmann-Krause bounded confidence
    rng = np.random.default_rng(seed)
    opinions = np.zeros((steps, n_agents), dtype=np.float64)
    opinions[0] = rng.uniform(0.0, 1.0, size=n_agents)

    for t in range(steps - 1):
        curr = opinions[t]
        diff_mat = np.abs(curr[:, None] - curr[None, :])
        in_bound = (diff_mat <= confidence_eps).astype(np.float64)
        counts = np.sum(in_bound, axis=1)
        opinions[t + 1] = in_bound.dot(curr) / np.maximum(counts, 1.0)

    final_sorted = np.sort(opinions[-1])
    n_clusters = 1 + int(np.sum(np.diff(final_sorted) > 0.03))
    return opinions, n_clusters

# ------------------------------------------------------------------------------
# 3. BRIAN ARTHUR'S EL FAROL BAR (1994) & CHALLET-ZHANG MINORITY GAME (1997)
# ------------------------------------------------------------------------------

def simulate_el_farol_bar(n_agents=100, capacity=60, memory_m=4, n_strategies=6, weeks=110, seed=42):
    # Inductive predictor competition on shared attendance history (Arthur, 1994)
    rng = np.random.default_rng(seed)
    history = list(rng.integers(35, 85, size=10))
    # Generate linear autoregressive predictor weights + bias for each agent and strategy
    weights = rng.uniform(-0.8, 0.8, size=(n_agents, n_strategies, memory_m))
    biases = rng.uniform(20.0, 80.0, size=(n_agents, n_strategies))
    attendance_series = []

    for w in range(weeks):
        recent = np.array(history[-memory_m:], dtype=np.float64)
        # Evaluate candidate predictions for current week
        preds = np.tensordot(weights, recent, axes=([2], [0])) + biases
        preds = np.clip(preds, 0.0, float(n_agents))

        # Score strategies on past 4 weeks of history
        strat_errors = np.zeros((n_agents, n_strategies), dtype=np.float64)
        for back in range(1, 5):
            past_window = np.array(history[-(memory_m + back) : -back], dtype=np.float64)
            actual_val = float(history[-back])
            past_preds = np.tensordot(weights, past_window, axes=([2], [0])) + biases
            strat_errors += np.abs(past_preds - actual_val)

        best_strat = np.argmin(strat_errors, axis=1)
        chosen_forecast = preds[np.arange(n_agents), best_strat]
        decisions = (chosen_forecast < capacity).astype(np.int32)
        total_att = int(np.sum(decisions))
        attendance_series.append(total_att)
        history.append(total_att)

    return np.array(attendance_series)

def simulate_minority_game(n_agents=101, memory_m=3, herding_coupling=0.0, steps=180, seed=42):
    # Challet & Zhang (1997) Minority Game + optional Workshop Networked Herding Extension
    rng = np.random.default_rng(seed)
    n_states = 1 << memory_m
    # Each agent has S=2 lookup tables mapping history state -> {-1, +1}
    strategies = rng.choice([-1, 1], size=(n_agents, 2, n_states))
    scores = np.zeros((n_agents, 2), dtype=np.float64)
    state = int(rng.integers(0, n_states))
    excess_demand = []

    for _ in range(steps):
        chosen_s = (scores[:, 1] > scores[:, 0]).astype(np.int32)
        actions = strategies[np.arange(n_agents), chosen_s, state].copy()

        # Workshop Extension: Local neighbor imitation on a 1D ring when herding_coupling > 0
        if herding_coupling > 0.0:
            imitate_mask = rng.random(n_agents) < herding_coupling
            neighbor_actions = np.roll(actions, 1)
            actions[imitate_mask] = neighbor_actions[imitate_mask]

        A_t = int(np.sum(actions))
        excess_demand.append(A_t)
        minority_sign = -1 if A_t > 0 else 1

        # Update virtual strategy scores
        for s_idx in (0, 1):
            scores[:, s_idx] += strategies[:, s_idx, state] * minority_sign

        winning_bit = 1 if minority_sign == 1 else 0
        state = ((state << 1) | winning_bit) & (n_states - 1)

    norm_volatility = float(np.var(excess_demand)) / float(n_agents)
    alpha = float(n_states) / float(n_agents)
    return np.array(excess_demand), norm_volatility, alpha

# ------------------------------------------------------------------------------
# 4. INTERACTIVE MODULE 07 DASHBOARD
# ------------------------------------------------------------------------------

def run_module07_lab(study='1. Diversity Prediction Theorem', param_val=0.25, herding_ext=False):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    fig.patch.set_facecolor('#F8FAFC')

    if study == '1. Diversity Prediction Theorem':
        rho_val = min(max(param_val, 0.0), 0.95)
        e_coll, e_indiv, div_pred, exact_tuple = simulate_dpt_ensemble(50, 10.0, rho_val, 400, 42)

        # Panel 1: Theoretical vs Monte Carlo Collective Error across Error Correlation rho
        rhos = np.linspace(0.0, 0.95, 20)
        n_ag, sig = 50, 10.0
        theory_coll = (sig ** 2) * (1.0 / n_ag + (1.0 - 1.0 / n_ag) * rhos)
        emp_coll = [simulate_dpt_ensemble(n_ag, sig, r, 250, 42)[0] for r in rhos]

        axes[0].set_facecolor('#F8FAFC')
        axes[0].plot(rhos, theory_coll, color='#2563EB', linewidth=2.4, label='Ex-Ante Theory: E[E_coll]')
        axes[0].scatter(rhos, emp_coll, color='#E11D48', s=36, zorder=5, label='Monte Carlo Ensemble')
        axes[0].axhline(sig ** 2, color='#64748B', linestyle='--', label='Single Member Expected Error (sigma^2)')
        axes[0].set_title('1. Impact of Error Correlation rho on Collective Error', fontsize=11, fontweight='bold')
        axes[0].set_xlabel('Pairwise Error Correlation rho')
        axes[0].set_ylabel('Mean Squared Error')
        axes[0].legend(facecolor='white')
        axes[0].grid(True, linestyle='--', alpha=0.4)

        # Panel 2: Bar decomposition at selected rho
        axes[1].set_facecolor('#F8FAFC')
        labels = ['Mean Individual\nError E_indiv', 'Collective Error\nE_collective', 'Prediction\nDiversity D']
        vals = [e_indiv, e_coll, div_pred]
        bars = axes[1].bar(labels, vals, color=['#64748B', '#2563EB', '#0D9488'], width=0.5, edgecolor='#0F172A')
        for b in bars:
            h = b.get_height()
            axes[1].text(b.get_x() + b.get_width()/2, h + 2.0, f'{h:.1f}', ha='center', fontweight='bold')
        axes[1].set_ylim(0, 125)
        axes[1].set_title(f'2. Error Decomposition at rho = {rho_val:.2f}', fontsize=11, fontweight='bold')
        axes[1].set_ylabel('Squared Error Units')
        axes[1].grid(axis='y', linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.show()
        ec0, ei0, dp0 = exact_tuple
        print(f'Single-Trial Algebraic Identity Check: E_indiv ({ei0:.2f}) - Diversity ({dp0:.2f}) = E_coll ({ec0:.2f})')

    elif study == '2. Bounded Confidence (HK vs DeGroot)':
        eps = max(0.04, min(0.50, param_val))
        opinions, n_clus = simulate_hk_opinions(n_agents=80, confidence_eps=eps, steps=30, seed=42)
        axes[0].set_facecolor('#F8FAFC')
        for i in range(80):
            axes[0].plot(opinions[:, i], color='#2563EB', alpha=0.35, linewidth=1.1)
        axes[0].set_ylim(-0.02, 1.02)
        axes[0].set_title(f'1. Hegselmann-Krause Trajectories (eps={eps:.2f} | Clusters={n_clus})', fontsize=11, fontweight='bold')
        axes[0].set_xlabel('Interaction Step t')
        axes[0].set_ylabel('Agent Opinion x_i(t)')
        axes[0].grid(True, linestyle='--', alpha=0.4)

        eps_grid = np.linspace(0.05, 0.45, 17)
        clus_counts = [simulate_hk_opinions(80, e, 30, 42)[1] for e in eps_grid]
        axes[1].set_facecolor('#F8FAFC')
        axes[1].plot(eps_grid, clus_counts, marker='o', color='#D97706', linewidth=2.2, label='Hegselmann-Krause')
        axes[1].axhline(1.0, color='#0D9488', linestyle='--', linewidth=1.8, label='DeGroot Linear Consensus (1 Cluster)')
        axes[1].set_title('2. Opinion Fragmentation & Polarization Bifurcation', fontsize=11, fontweight='bold')
        axes[1].set_xlabel('Confidence Bound epsilon')
        axes[1].set_ylabel('Final Number of Opinion Clusters')
        axes[1].legend(facecolor='white')
        axes[1].grid(True, linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.show()
        print(f'HK Bounded Confidence eps={eps:.2f} -> Converged into {n_clus} distinct opinion cluster(s).')

    else:
        # Mode 3: Brian Arthur El Farol Bar & Challet-Zhang Minority Game
        mem_m = int(round(2 + param_val * 4))
        att = simulate_el_farol_bar(n_agents=100, capacity=60, memory_m=mem_m, weeks=100, seed=42)
        axes[0].set_facecolor('#F8FAFC')
        axes[0].plot(att, color='#2563EB', linewidth=1.8, label='Weekly Attendance A(t)')
        axes[0].axhline(60, color='#E11D48', linestyle='--', linewidth=1.8, label='Comfort Threshold (L = 60)')
        axes[0].set_ylim(15, 100)
        axes[0].set_title(f'1. Arthur (1994) El Farol Bar (Mean={np.mean(att):.1f}, SD={np.std(att):.1f})', fontsize=11, fontweight='bold')
        axes[0].set_xlabel('Week t')
        axes[0].set_ylabel('Bar Attendance')
        axes[0].legend(facecolor='white')
        axes[0].grid(True, linestyle='--', alpha=0.4)

        # Minority Game Volatility Phase Curve across memory M
        m_vals = [1, 2, 3, 4, 5, 6]
        alphas, vols_indep, vols_herd = [], [], []
        for m in m_vals:
            _, v0, a0 = simulate_minority_game(101, memory_m=m, herding_coupling=0.0, steps=160, seed=42)
            _, vh, _ = simulate_minority_game(101, memory_m=m, herding_coupling=0.35, steps=160, seed=42)
            alphas.append(a0)
            vols_indep.append(v0)
            vols_herd.append(vh)

        axes[1].set_facecolor('#F8FAFC')
        axes[1].loglog(alphas, vols_indep, marker='o', color='#0D9488', linewidth=2.2, label='Original MG (Independent Agents)')
        if herding_ext:
            axes[1].loglog(alphas, vols_herd, marker='s', color='#E11D48', linewidth=2.2, linestyle='--', label='[Extension] Networked Herding (q=0.35)')
        axes[1].axhline(1.0, color='#64748B', linestyle=':', label='Random Coin-Flip Baseline (sigma^2/N = 1)')
        axes[1].set_title('2. Minority Game Volatility Phase Diagram (Challet & Zhang)', fontsize=11, fontweight='bold')
        axes[1].set_xlabel('Control Parameter alpha = 2^M / N (Log Scale)')
        axes[1].set_ylabel('Normalized Volatility sigma^2 / N (Log Scale)')
        axes[1].legend(facecolor='white')
        axes[1].grid(True, which='both', linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.show()
        print(f'El Farol Mean Attendance = {np.mean(att):.2f} (Threshold=60) | MG Memory M={mem_m} (alpha={2**mem_m/101.0:.3f})')

widgets.interact(
    run_module07_lab,
    study=widgets.Dropdown(options=['1. Diversity Prediction Theorem', '2. Bounded Confidence (HK vs DeGroot)', '3. El Farol Bar & Minority Game'], value='1. Diversity Prediction Theorem', description='Model Lab:'),
    param_val=widgets.FloatSlider(min=0.05, max=0.90, step=0.05, value=0.25, description='Param (rho/eps):'),
    herding_ext=widgets.Checkbox(value=False, description='Enable Networked Herding Extension'),
);
